# 🌊 Water Quality Monitor — Copernicus JupyterHub Notebook

This notebook connects to the Copernicus openEO backend and:
- Fetches Sentinel-2 L2A data for any coordinate
- Computes NDWI (water detection) and NDCI (pollution proxy)
- Visualizes the results on a map
- Can be used standalone OR as a data exploration companion to the FastAPI backend

**Run this on:** https://jupyterhub.dataspace.copernicus.eu

## 1. Install dependencies

In [ ]:
# Run once to install required packages in the JupyterHub environment
import subprocess
subprocess.run(['pip', 'install', 'openeo', 'rasterio', 'matplotlib', 'numpy', 'folium', 'requests', '--quiet'])

## 2. Connect to Copernicus openEO

In [ ]:
import openeo
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Connect to Copernicus Data Space openEO
conn = openeo.connect('https://openeo.dataspace.copernicus.eu')

# Authenticate — this will prompt you to log in via browser on first run
# Credentials are cached in ~/.config/openeo/ after first authentication
conn.authenticate_oidc()

print('✅ Connected to Copernicus openEO!')
print(conn.describe_account())

## 3. Configure Target Location

In [ ]:
# ── CHANGE THESE COORDINATES to your area of interest ──────────────────────────
# Default: Lake Ohrid, Macedonia (a UNESCO World Heritage water body)
LAT = 41.0297   # Latitude
LON = 20.7169   # Longitude

# Bounding box offset in degrees (~1km radius)
OFFSET = 0.01

# How many days back to look for imagery
DAYS_BACK = 10

# Compute derived values
bbox = {'west': LON-OFFSET, 'south': LAT-OFFSET, 'east': LON+OFFSET, 'north': LAT+OFFSET}
end_date = datetime.utcnow().strftime('%Y-%m-%d')
start_date = (datetime.utcnow() - timedelta(days=DAYS_BACK)).strftime('%Y-%m-%d')

print(f'📍 Location: lat={LAT}, lon={LON}')
print(f'📦 Bounding box: {bbox}')
print(f'🗓️  Time range: {start_date} → {end_date}')

## 4. Fetch Sentinel-2 Data & Compute Indices

In [ ]:
import io
import rasterio

BANDS = ['B02', 'B03', 'B04', 'B05', 'B08']

# Load Sentinel-2 L2A collection
datacube = conn.load_collection(
    'SENTINEL2_L2A',
    spatial_extent=bbox,
    temporal_extent=[start_date, end_date],
    bands=BANDS + ['SCL'],
    max_cloud_cover=80,
)

# Cloud masking via SCL (Scene Classification Layer)
# Keep only vegetation (4), non-vegetated (5), water (6), unclassified (7)
scl = datacube.band('SCL')
valid_mask = (scl == 4) | (scl == 5) | (scl == 6) | (scl == 7)
datacube_masked = datacube.mask(~valid_mask).filter_bands(BANDS)

# Temporal median composite → one cloud-reduced image
composite = datacube_masked.reduce_dimension(dimension='t', reducer='median')

# Normalize reflectance (Sentinel-2 stores as DN 0–10000)
normalized = composite / 10000.0

print('⏳ Submitting job to openEO... (this takes ~1-2 minutes for small areas)')
result_bytes = normalized.download(format='GTiff')
print('✅ Download complete!')

# Parse GeoTIFF into band arrays
bands = {}
with rasterio.open(io.BytesIO(result_bytes)) as src:
    for i, name in enumerate(BANDS, start=1):
        arr = src.read(i).astype(np.float32)
        nodata = src.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan
        bands[name] = arr
    transform = src.transform
    crs = src.crs

print(f'📊 Band shape: {bands["B03"].shape}')

## 5. Compute Water Quality Indices

In [ ]:
def safe_nd(a, b):
    """Normalized difference (A-B)/(A+B) with NaN guard."""
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where((a + b) == 0, np.nan, (a - b) / (a + b)).astype(np.float32)

# NDWI = (Green - NIR) / (Green + NIR)  -> positive = water
ndwi = safe_nd(bands['B03'], bands['B08'])

# NDCI = (RedEdge - Red) / (RedEdge + Red)  -> higher = more algae/chlorophyll
ndci = safe_nd(bands['B05'], bands['B04'])

trb = bands['B04'] / bands['B03']
ssd = bands['B04'] / bands['B02']

# Scalar statistics (median over the spatial extent)
ndwi_val = float(np.nanmedian(ndwi))
ndci_val = float(np.nanmedian(ndci))
trb_val  = float(np.nanmedian(trb))
ssd_val  = float(np.nanmedian(ssd))

water_detected = ndwi_val > 0.1

if ndci_val < 0:
    pollution = 'LOW'
elif ndci_val <= 0.2:
    pollution = 'MEDIUM'
else:
    pollution = 'HIGH'

# ERA5 Rainfall (last 5 days)
# Reuses the same helper as the FastAPI backend for consistency.
# Falls back gracefully to None if ERA5 is unavailable.
try:
    import sys, os
    sys.path.insert(0, os.path.dirname(os.path.abspath('data_fetch.py')))
    from data_fetch import fetch_era5_rainfall, get_bounding_box, connect_to_openeo
    _conn = connect_to_openeo()
    _bbox = get_bounding_box(LAT, LON)
    rainfall_mm = fetch_era5_rainfall(_conn, _bbox, days_back=5)
except Exception as _e:
    print(f'ERA5 rainfall fetch failed (non-fatal): {_e}')
    rainfall_mm = None

# Classify rainfall impact
if rainfall_mm is None:
    rainfall_label = 'UNKNOWN'
elif rainfall_mm < 5:
    rainfall_label = 'DRY'
elif rainfall_mm < 20:
    rainfall_label = 'MODERATE'
elif rainfall_mm < 50:
    rainfall_label = 'HIGH'
else:
    rainfall_label = 'EXTREME'

print('=' * 50)
print(f'Location:             lat={LAT}, lon={LON}')
print(f'NDWI:                 {ndwi_val:.4f}  (>0.1 = water)')
print(f'Chlorophyll (NDCI):   {ndci_val:.4f}  (>0.2 = algae risk)')
print(f'Turbidity:            {trb_val:.4f}')
print(f'Suspended sediment:   {ssd_val:.4f}')
print(f'Water detected:       {water_detected}')
print(f'Pollution:            {pollution}')
print(f'Rainfall (5-day):     {rainfall_mm if rainfall_mm is not None else "N/A"} mm  [{rainfall_label}]')
print('=' * 50)


## 6. Visualize Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'Sentinel-2 Water Quality — lat={LAT}, lon={LON}', fontsize=14, fontweight='bold')

# RGB True color (B04=Red, B03=Green, B02 not loaded so use B03 as proxy)
# Use B04, B03, B03 for a rough visualization if B02 not available
rgb = np.stack([
    np.clip(bands['B04'] * 3.5, 0, 1),
    np.clip(bands['B03'] * 3.5, 0, 1),
    np.clip(bands['B03'] * 3.5, 0, 1),
], axis=-1)
axes[0].imshow(rgb)
axes[0].set_title('True Color (RGB approx.)')
axes[0].axis('off')

# NDWI map
im1 = axes[1].imshow(ndwi, cmap='RdYlBu', vmin=-1, vmax=1)
plt.colorbar(im1, ax=axes[1], fraction=0.046)
axes[1].set_title(f'NDWI (median={ndwi_val:.3f})')
axes[1].axis('off')

# NDCI map
im2 = axes[2].imshow(ndci, cmap='YlOrRd', vmin=-0.5, vmax=0.5)
plt.colorbar(im2, ax=axes[2], fraction=0.046)
axes[2].set_title(f'NDCI — {pollution}')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('water_quality_output.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved as water_quality_output.png')

## 7. (Optional) Call the FastAPI Backend from the Notebook

In [ ]:
import requests
import json

# ── Set this to your running FastAPI server URL ─────────────────────────────────
# If running locally:       http://localhost:8000
# If deployed (e.g. ngrok): https://your-ngrok-url.ngrok.io
API_URL = 'http://localhost:8000'

payload = {'lat': LAT, 'lon': LON}

try:
    response = requests.post(f'{API_URL}/analyze-water', json=payload, timeout=300)
    response.raise_for_status()
    result = response.json()
    print('✅ API Response:')
    print(json.dumps(result, indent=2))
except requests.exceptions.ConnectionError:
    print('⚠️  Could not reach the API. Make sure the FastAPI server is running.')
    print('   Start it with: uvicorn api:app --host 0.0.0.0 --port 8000')
except Exception as e:
    print(f'❌ Error: {e}')

## 8. Batch Analysis — Multiple Locations

Useful for testing several Macedonian water bodies at once.

In [ ]:
import concurrent.futures
import time

# Key Macedonian water bodies
LOCATIONS = [
    {'name': 'Lake Ohrid',   'lat': 41.0297, 'lon': 20.7169},
    {'name': 'Lake Prespa',  'lat': 40.8500, 'lon': 20.9500},
    {'name': 'Lake Dojran',  'lat': 41.2000, 'lon': 22.7300},
    {'name': 'Vardar River', 'lat': 41.9961, 'lon': 21.4320},
]

API_URL = 'http://localhost:8000'  # update if needed

def analyze_location(loc):
    try:
        r = requests.post(f'{API_URL}/analyze-water', json={'lat': loc['lat'], 'lon': loc['lon']}, timeout=300)
        result = r.json()
        return {**loc, **result}
    except Exception as e:
        return {**loc, 'error': str(e)}

print('🔄 Running batch analysis (parallel)...')
start = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:
    results = list(pool.map(analyze_location, LOCATIONS))

print(f'✅ Done in {time.time()-start:.1f}s\n')

for r in results:
    name = r.get('name', 'Unknown')
    if 'error' in r:
        print(f'❌ {name}: {r["error"]}')
    else:
        icon = '💧' if r.get('water_detected') else '🏔️'
        print(f'{icon} {name:15s}  NDWI={r.get("ndwi","N/A"):>7}  NDCI={r.get("ndci","N/A"):>7}  Pollution: {r.get("pollution_status","N/A")}')